# CropCheckUp — BiRefNet background removal

Removes the background from every PlantVillage image in `rasagyavatsal/cropcheckup-dataset` with
**BiRefNet-General** (`ZhengPeng7/BiRefNet`) and saves transparent PNGs.

Output (in `/kaggle/working`):
- `segmented__<class>.zip` — one per class, each containing `segmented/<class>/<name>.png` (RGBA)
- `manifest.json` — classes completed, image counts, failures, fallbacks, throughput
- `preview.png` — a few results for a quick visual check

One worker process per GPU, each with its own share of the classes. (`DataParallel` across Kaggle's 2x T4
died with an NCCL error mid-run; independent processes never communicate, so that can't recur.) Each class is
zipped as soon as it finishes, so a run that hits the time budget or loses a worker keeps every completed class.

BiRefNet finds no subject in close-ups where the leaf fills the frame (common for corn) and returns an empty or
lesion-only mask. Masks covering less than `MIN_COVERAGE` fall back to the full original image, fully opaque;
those files are listed under `fallback_full_image` in the manifest.

In [ ]:
import subprocess, sys

# BiRefNet's remote code doesn't load under transformers 5.x.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40,<5", "timm", "kornia", "einops"])

In [ ]:
import json
from pathlib import Path

import torch

# Kaggle's mount layout under /kaggle/input has changed over time, so locate raw/ instead of hard-coding it.
RAW_DIR = next(p for p in Path("/kaggle/input").rglob("raw") if p.parent.name == "plantvillage")
WORK_DIR = Path("/kaggle/working")
MODEL_ID = "ZhengPeng7/BiRefNet"
IMG_SIZE = 1024
BATCH_SIZE = 8  # per GPU
MIN_COVERAGE = 0.10  # below this foreground fraction, keep the whole image instead of the mask
SKIP_CLASSES = []  # classes already done in an earlier run
LIMIT_PER_CLASS = None  # cap images per class (timing runs); None = all
TIME_BUDGET_H = 10  # don't start a new class after this; the largest class takes ~45 min, Kaggle's limit is 12h
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
N_GPUS = max(torch.cuda.device_count(), 1)
print(RAW_DIR, N_GPUS, "GPU(s)")

In [ ]:
# Split classes across GPUs, largest first onto the least-loaded GPU, so both finish at about the same time.
sizes = {
    d.name: min(sum(p.suffix.lower() in IMAGE_EXTS for p in d.iterdir()), LIMIT_PER_CLASS or 10**9)
    for d in RAW_DIR.iterdir() if d.is_dir() and d.name not in SKIP_CLASSES
}
assignments, loads = [[] for _ in range(N_GPUS)], [0] * N_GPUS
for cls in sorted(sizes, key=sizes.get, reverse=True):
    g = loads.index(min(loads))
    assignments[g].append(cls)
    loads[g] += sizes[cls]
print(len(sizes), "classes,", sum(sizes.values()), "images; per GPU:", loads)

(WORK_DIR / "config.json").write_text(json.dumps({
    "raw_dir": str(RAW_DIR), "work_dir": str(WORK_DIR), "model_id": MODEL_ID, "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE, "min_coverage": MIN_COVERAGE, "limit_per_class": LIMIT_PER_CLASS,
    "time_budget_h": TIME_BUDGET_H, "image_exts": sorted(IMAGE_EXTS), "assignments": assignments,
}))

# Download weights and remote code once here, so the workers don't race on the Hugging Face cache.
from transformers import AutoModelForImageSegmentation
AutoModelForImageSegmentation.from_pretrained(MODEL_ID, trust_remote_code=True)
None  # don't render the model repr

In [ ]:
%%writefile /kaggle/working/worker.py
import io, json, sys, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

rank = int(sys.argv[1])
cfg = json.loads(Path("/kaggle/working/config.json").read_text())
RAW_DIR, WORK_DIR = Path(cfg["raw_dir"]), Path(cfg["work_dir"])
classes = cfg["assignments"][rank]

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForImageSegmentation.from_pretrained(cfg["model_id"], trust_remote_code=True).to(device, dtype).eval()

transform = transforms.Compose([
    transforms.Resize((cfg["img_size"], cfg["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
to_pil = transforms.ToPILImage()


class Images(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return transform(Image.open(self.paths[i]).convert("RGB")), i


def cutout_png(path, pred):
    # Returns (PNG bytes, whether the mask was too small and the full image was kept instead).
    image = Image.open(path).convert("RGB")
    fallback = (pred > 0.5).float().mean().item() < cfg["min_coverage"]
    mask = Image.new("L", image.size, 255) if fallback else to_pil(pred).resize(image.size, Image.BILINEAR)
    image.putalpha(mask)
    # Zero the colour under fully transparent pixels so the background can't resurface via .convert("RGB").
    rgba = np.array(image)
    rgba[rgba[..., 3] == 0, :3] = 0
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format="PNG", optimize=True)
    return buf.getvalue(), fallback


manifest = {"completed": {}, "skipped_for_time": [], "failures": [], "fallback_full_image": []}
manifest_path = WORK_DIR / f"manifest_gpu{rank}.json"
start, n_done = time.time(), 0
saver = ThreadPoolExecutor(2)
for ci, cls in enumerate(classes):
    if (time.time() - start) / 3600 > cfg["time_budget_h"]:
        manifest["skipped_for_time"] = classes[ci:]
        print(f"[gpu{rank}] time budget reached; not started: {classes[ci:]}", flush=True)
        break

    exts = set(cfg["image_exts"])
    paths = sorted(p for p in (RAW_DIR / cls).iterdir() if p.suffix.lower() in exts)[:cfg["limit_per_class"]]
    loader = torch.utils.data.DataLoader(Images(paths), batch_size=cfg["batch_size"], num_workers=2)
    pending = []
    for x, idx in loader:
        with torch.no_grad():
            preds = model(x.to(device, dtype))[-1].sigmoid().float().cpu()
        pending += [(paths[i], saver.submit(cutout_png, paths[i], pred.squeeze())) for pred, i in zip(preds, idx.tolist())]

    # Write to a temp name and rename, so a crash mid-class never leaves a truncated zip behind.
    tmp = WORK_DIR / f"segmented__{cls}.zip.part"
    saved = 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:  # PNGs are already compressed
        for path, fut in pending:
            try:
                png, fallback = fut.result()
            except Exception as e:
                manifest["failures"].append({"path": str(path), "error": repr(e)})
                continue
            zf.writestr(f"segmented/{cls}/{path.stem}.png", png)
            saved += 1
            if fallback:
                manifest["fallback_full_image"].append(f"{cls}/{path.name}")
    tmp.rename(WORK_DIR / f"segmented__{cls}.zip")

    manifest["completed"][cls] = {"input": len(paths), "saved": saved}
    manifest_path.write_text(json.dumps(manifest, indent=1))
    n_done += len(paths)
    elapsed = time.time() - start
    print(f"[gpu{rank} {ci + 1}/{len(classes)}] {cls}: {saved}/{len(paths)}  |  {n_done / elapsed:.2f} img/s, {elapsed / 60:.1f} min elapsed", flush=True)

saver.shutdown()
manifest["images_per_second"] = n_done / (time.time() - start)
manifest_path.write_text(json.dumps(manifest, indent=1))

In [ ]:
import os, threading

def forward(proc):
    for line in proc.stdout:
        print(line, end="", flush=True)

procs = []
for rank in range(N_GPUS):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(rank)}
    proc = subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(rank)], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=forward, args=(proc,), daemon=True).start()
    procs.append(proc)
codes = [p.wait() for p in procs]
print("worker exit codes:", codes)

# Merge per-worker manifests. A worker that crashed still leaves the classes it finished.
manifest = {"completed": {}, "skipped_for_time": [], "failures": [], "fallback_full_image": [], "not_run": []}
for rank, assigned in enumerate(assignments):
    path = WORK_DIR / f"manifest_gpu{rank}.json"
    part = json.loads(path.read_text()) if path.exists() else {}
    for key in ("completed", "skipped_for_time", "failures", "fallback_full_image"):
        if key in part:
            manifest[key] = {**manifest[key], **part[key]} if key == "completed" else manifest[key] + part[key]
    manifest["not_run"] += [c for c in assigned if c not in part.get("completed", {}) and c not in part.get("skipped_for_time", [])]
    manifest[f"gpu{rank}_images_per_second"] = part.get("images_per_second")
    path.unlink(missing_ok=True)
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1))
for f in ("config.json", "worker.py"):
    (WORK_DIR / f).unlink()
for leftover in WORK_DIR.glob("*.zip.part"):
    leftover.unlink()

print(len(manifest["completed"]), "classes done,", sum(v["saved"] for v in manifest["completed"].values()), "images")
print(json.dumps({k: v for k, v in manifest.items() if k not in ("completed", "fallback_full_image")}, indent=1))
print(len(manifest["fallback_full_image"]), "images kept whole (mask below MIN_COVERAGE)")
if any(codes):
    raise RuntimeError(f"worker(s) failed: {codes}; completed classes are saved, rerun with SKIP_CLASSES")

In [ ]:
import zipfile
import matplotlib.pyplot as plt
from PIL import Image

zf = zipfile.ZipFile(next(WORK_DIR.glob("segmented__*.zip")))
names = zf.namelist()[:8]
fig, axes = plt.subplots(2, len(names), figsize=(2.2 * len(names), 4.6))
for col, name in enumerate(names):
    cls, stem = name.split("/")[1], Path(name).stem
    rgba = Image.open(zf.open(name))
    on_black = Image.new("RGB", rgba.size, (0, 0, 0))
    on_black.paste(rgba, mask=rgba.split()[-1])
    for row, img in enumerate((Image.open(next((RAW_DIR / cls).glob(f"{stem}.*"))), on_black)):
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
    axes[0, col].set_title(cls[:22], fontsize=7)
plt.tight_layout()
fig.savefig(WORK_DIR / "preview.png", dpi=90)
plt.show()